# Cell 1: setup

In [ ]:
import os, glob, json, re, time, threading
import numpy as np
import pandas as pd

SHARD, NSHARDS = 0, 2
LIMIT = None

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
OUT = "/kaggle/working/labels_raw.jsonl"

train = pd.read_csv(f"{ROOT}/train.csv")
order = np.argsort(train["Report"].str.len().values, kind="stable")
mine = train.iloc[order[SHARD::NSHARDS]].reset_index(drop=True)
if LIMIT:
    mine = mine.iloc[::max(1, len(mine) // LIMIT)].head(LIMIT).reset_index(drop=True)
print("shard", SHARD, "studies:", len(mine))

# Cell 2: load Qwen3-8B in 4-bit, one copy per GPU

In [ ]:
!pip install -q bitsandbytes
import torch
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

cfgs = [p for top in os.listdir("/kaggle/input") if top != "competitions"
        for p in glob.glob(f"/kaggle/input/{top}/**/config.json", recursive=True)]
DIR_8B = [os.path.dirname(c) for c in cfgs if "/8b/" in c][0]
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16)
toks, models = [], []
for i in range(torch.cuda.device_count()):
    t = AutoTokenizer.from_pretrained(DIR_8B)
    t.padding_side = "left"
    toks.append(t)
    models.append(AutoModelForCausalLM.from_pretrained(DIR_8B, quantization_config=bnb,
                                                       device_map={"": i}).eval())
print(DIR_8B, {i: f"{torch.cuda.memory_allocated(i) / 1e9:.1f} GB" for i in range(len(models))})

# Cell 3: prompt, parser, generation

In [ ]:
SYSTEM = """You read knee MRI radiology reports and extract 12 findings for one knee.
Reports can be in any language. Answer only with JSON, no other text.

For each finding return [severity, certainty, evidence].
severity:
  0 = the report explicitly says it is normal, intact or absent; quote those words
  1 = minimal, trace, degenerative, intrasubstance signal, chronic or old
  2 = mild, small, low grade, grade 1
  3 = moderate, partial tear, grade 2, or a tear or finding that is present with no severity words
  4 = severe, large, massive, complete or full-thickness, grade 3 or 4, displaced
  null = the report does not mention this finding; never use 0 for an unmentioned finding
certainty: 2 = definite, 1 = probable or likely, 0 = possible, suspected, cannot be excluded
evidence: at most 8 words copied from the report in its original language, or ""

Findings:
ACL, MCL: injury or tear of that ligament.
Medial Meniscus, Lateral Meniscus: tear or abnormality of that meniscus. Signal that does not
  reach the surface is severity 1.
Medial OA, Lateral OA: cartilage loss of the femoral condyle or tibial plateau on that side.
PF OA: cartilage loss of the patella or trochlea.
Effusion: joint fluid. Synovitis: synovial thickening or inflammation.
Baker's: popliteal cyst. Contusion: bone bruise or marrow edema without a fracture line.
Fracture: fracture line or cortical break. An old or healed fracture is severity 1.

If the report covers both knees, use the knee the report is mainly about.
Return one JSON object with exactly these keys: """ + ", ".join(f'"{k}"' for k in LABELS) + """
Example value: "ACL": [4, 2, "complete tear of the ACL"]"""

def prompt(tok, report):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": "Report:\n" + report}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)

def as_int(v):
    try:
        return int(v)
    except (TypeError, ValueError):
        return None

def parse(text):
    t = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        d = json.loads(t)
        out = {}
        for k in LABELS:
            v = d.get(k)
            ok = isinstance(v, list) and len(v) >= 2
            out[k] = (as_int(v[0]), as_int(v[1]), v[2] if len(v) > 2 else "") if ok else (None, None, "")
        return out, "json", 12
    except Exception:
        pass
    out, hits = {}, 0
    for k in LABELS:
        m = re.search('"' + re.escape(k) + r'"\s*:\s*\[\s*(null|-?\d+)\s*,\s*(null|-?\d+)'
                      r'(?:\s*,\s*"((?:[^"\\]|\\.)*)")?', t)
        if m:
            hits += 1
            out[k] = (as_int(m.group(1)), as_int(m.group(2)), m.group(3) or "")
        else:
            out[k] = (None, None, "")
    return (out, "regex", hits) if hits else (None, "fail", 0)

@torch.no_grad()
def gen_batch(model, tok, texts, max_new_tokens):
    enc = out = None
    try:
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    except torch.OutOfMemoryError:
        if len(texts) == 1:
            raise
    enc = out = None
    with torch.cuda.device(model.device):
        torch.cuda.empty_cache()
    h = len(texts) // 2
    return gen_batch(model, tok, texts[:h], max_new_tokens) + gen_batch(model, tok, texts[h:], max_new_tokens)

lock = threading.Lock()

def label_pass(df, bs, max_new_tokens, tag):
    batches = [df.iloc[i:i + bs] for i in range(0, len(df), bs)]
    state = {"done": 0, "t0": time.time()}
    def worker(w):
        m, t = models[w], toks[w]
        for b in batches[w::len(models)]:
            try:
                outs = gen_batch(m, t, [prompt(t, r) for r in b["Report"]], max_new_tokens)
            except Exception as err:
                print(f"worker {w} stopped: {err!r}"[:300])
                return
            with lock:
                with open(OUT, "a") as f:
                    for sid, o in zip(b["StudyInstanceUID"], outs):
                        f.write(json.dumps({"StudyInstanceUID": sid, "raw": o, "pass": tag}) + "\n")
                state["done"] += len(b)
                if state["done"] % (bs * 10) < bs:
                    el = time.time() - state["t0"]
                    print(f"[{tag}] {state['done']}/{len(df)} in {el / 60:.0f} min")
    with ThreadPoolExecutor(len(models)) as ex:
        list(ex.map(worker, range(len(models))))

# Cell 4: main pass, then retries

In [ ]:
def load_best():
    best = {}
    if not os.path.exists(OUT):
        return best
    rank = {"json": 2, "regex": 1, "fail": 0}
    for line in open(OUT):
        r = json.loads(line)
        p, status, hits = parse(r["raw"])
        key = (rank[status], hits)
        if r["StudyInstanceUID"] not in best or key > best[r["StudyInstanceUID"]][0]:
            best[r["StudyInstanceUID"]] = (key, p, status, r["raw"])
    return best

t0 = time.time()
best = load_best()
todo = mine[~mine["StudyInstanceUID"].isin(best)]
print("main pass:", len(todo), "to do")
label_pass(todo, bs=12, max_new_tokens=600, tag=1)

best = load_best()
retry = mine[mine["StudyInstanceUID"].map(lambda s: s not in best or best[s][2] != "json")]
print("retry pass:", len(retry), "to do")
if len(retry):
    label_pass(retry, bs=4, max_new_tokens=1000, tag=2)
print(f"total {(time.time() - t0) / 3600:.2f} h")

# Cell 5: final table and QA

In [ ]:
best = load_best()
rows = []
for sid, report in zip(mine["StudyInstanceUID"], mine["Report"]):
    if sid not in best:
        rows.append({"StudyInstanceUID": sid, "status": "missing"})
        continue
    _, p, status, _ = best[sid]
    row = {"StudyInstanceUID": sid, "status": status}
    rep = re.sub(r"\s+", " ", report).lower()
    for k in LABELS:
        sev, cert, ev = p[k] if p else (None, None, "")
        e = re.sub(r"\s+", " ", str(ev)).strip().lower()
        row[f"{k}|sev"], row[f"{k}|cert"], row[f"{k}|ev"] = sev, cert, ev
        row[f"{k}|ev_found"] = bool(e) and e in rep
    rows.append(row)

lab = pd.DataFrame(rows)
lab.to_csv(f"/kaggle/working/labels_shard{SHARD}.csv", index=False)
print(lab["status"].value_counts().to_string())
ev_cols = [c for c in lab.columns if c.endswith("|ev")]
has_ev = lab[ev_cols].fillna("").astype(str).apply(lambda s: s.str.strip() != "").values
found = lab[[c + "_found" for c in ev_cols]].values.astype(bool)
print(f"evidence found verbatim: {found[has_ev].mean():.1%} of {has_ev.sum()}")
sev = lab[[f"{k}|sev" for k in LABELS]]
sev.columns = LABELS
print(sev.apply(lambda s: s.value_counts(dropna=False)).fillna(0).astype(int).T.to_string())